# 6.01 扩散模型（DDPM）

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/06_generative_policy/6.01_ddpm.ipynb)


## 为什么要学

扩散模型是"会画画的 AI"（Stable Diffusion）背后的技术，也是 Diffusion Policy、Octo、RDT 等机器人策略的动作生成器。它的思想出奇地简单：

> **把数据一点点加噪声变成纯噪声很容易；学一个网络把这个过程反过来，就能从纯噪声"生成"数据。**

本课在二维点集上实现 DDPM（Ho et al. 2020），能直观地看到"从噪声中长出数据"的过程。

---

## 1. 前向过程：逐步加噪

给数据 $\mathbf x_0$ 加 $T$ 步高斯噪声，每步：

$$
q(\mathbf x_t|\mathbf x_{t-1})=\mathcal N\big(\sqrt{1-\beta_t}\,\mathbf x_{t-1},\ \beta_t I\big)
$$

$\beta_t$ 是事先定好的**噪声表**（noise schedule）。记 $\alpha_t=1-\beta_t$、$\bar\alpha_t=\prod_{s\le t}\alpha_s$，可以推出**一步到位**的公式（高斯的性质）：

$$
\boxed{\mathbf x_t=\sqrt{\bar\alpha_t}\,\mathbf x_0+\sqrt{1-\bar\alpha_t}\,\boldsymbol\epsilon,\qquad\boldsymbol\epsilon\sim\mathcal N(0,I)}
$$

$\bar\alpha_t$ 从 1 逐渐减小到接近 0：$t$ 小时 $\mathbf x_t$ 几乎就是数据，$t=T$ 时几乎是纯噪声。

## 2. 训练：预测噪声

DDPM 训练一个网络 $\boldsymbol\epsilon_\theta(\mathbf x_t,t)$ 来**预测加进去的噪声**：

$$
\mathcal L=\mathbb E_{\mathbf x_0,\boldsymbol\epsilon,t}\Big[\big\|\boldsymbol\epsilon-\boldsymbol\epsilon_\theta(\mathbf x_t,t)\big\|^2\Big]
$$

训练算法极其简单：随机取一个数据、一个时间步 $t$、一个噪声 $\boldsymbol\epsilon$，按上面的公式合成 $\mathbf x_t$，让网络猜 $\boldsymbol\epsilon$。

（这个简单的目标可以从变分下界严格推导出来，也等价于学习数据分布的**得分函数** $\nabla_{\mathbf x}\log p(\mathbf x_t)$，即"往数据密度高的方向走"。）

## 3. 采样：一步步去噪

从 $\mathbf x_T\sim\mathcal N(0,I)$ 出发，对 $t=T,\dots,1$：

$$
\mathbf x_{t-1}=\frac{1}{\sqrt{\alpha_t}}\left(\mathbf x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\boldsymbol\epsilon_\theta(\mathbf x_t,t)\right)+\sigma_t\mathbf z,\qquad\mathbf z\sim\mathcal N(0,I)
$$

其中 $\sigma_t^2=\tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t$（最后一步 $t=1$ 不加噪声）。

**为什么能表示多峰分布？** 每一步只需要预测一个**局部的、近似单峰的**去噪方向，而最终的随机性来自初始噪声 $\mathbf x_T$ 和每步的 $\mathbf z$：不同的初始噪声会"流向"不同的模式。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from vla_ladder.models.flow import cosine_beta_schedule, ChunkDenoiser
from vla_ladder.utils import set_seed, get_device

device = get_device()

def make_ring_of_gaussians(n, k=6, r=1.0, std=0.08, seed=0):
    """k 个高斯团均匀分布在圆环上：一个典型的多峰分布"""
    g = np.random.default_rng(seed)
    centers = np.stack([r * np.cos(2 * np.pi * np.arange(k) / k), r * np.sin(2 * np.pi * np.arange(k) / k)], 1)
    return torch.tensor(centers[g.integers(0, k, n)] + std * g.standard_normal((n, 2)), dtype=torch.float32)

data = make_ring_of_gaussians(20000).to(device)
T = 100
betas = cosine_beta_schedule(T).to(device)
alphas = 1 - betas
alphas_bar = torch.cumprod(alphas, 0)
plt.figure(figsize=(9, 2.5))
plt.subplot(1, 2, 1); plt.plot(alphas_bar.cpu()); plt.title("alpha_bar_t (cosine schedule)"); plt.xlabel("t")
plt.subplot(1, 2, 2); plt.scatter(*data[:3000].cpu().T, s=1); plt.gca().set_aspect("equal"); plt.title("data: 6 gaussians")
plt.tight_layout(); plt.show()

### 练习 1：一步加噪 `q_sample`

In [ ]:
def q_sample(x0, t, noise):
    """x0: (B, D)，t: (B,) 整数时间步 0..T-1，noise: (B, D) -> x_t"""
    # TODO: ab = alphas_bar[t] 变成 (B, 1)；返回 sqrt(ab) x0 + sqrt(1-ab) noise
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def q_sample(x0, t, noise):
    """x0: (B, D)，t: (B,) 整数时间步 0..T-1，noise: (B, D) -> x_t"""
    ab = alphas_bar[t][:, None]
    return ab.sqrt() * x0 + (1 - ab).sqrt() * noise

In [ ]:
# ✅ 自测 + 可视化前向过程
x0 = data[:2000]
fig, axes = plt.subplots(1, 5, figsize=(14, 2.8))
for ax, t in zip(axes, [0, 20, 50, 80, 99]):
    xt = q_sample(x0, torch.full((2000,), t, device=device), torch.randn_like(x0))
    ax.scatter(*xt.cpu().T, s=1); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal"); ax.set_title(f"t={t}")
plt.suptitle("forward process q(x_t | x_0)"); plt.tight_layout(); plt.show()
xT = q_sample(x0, torch.full((2000,), T - 1, device=device), torch.randn_like(x0))
assert abs(xT.std().item() - 1) < 0.1 and abs(xT.mean().item()) < 0.1     # t=T 时接近标准正态
print("通过 ✅")

### 练习 2：训练损失

我们用 `ChunkDenoiser`（第 6 章的通用去噪网络，见 `vla_ladder/models/flow.py`）作为 $\boldsymbol\epsilon_\theta$：把一个二维点看成"长度为 1 的动作块"，观测条件设成常数 0（无条件生成）。

In [ ]:
model = ChunkDenoiser(obs_dim=1, act_dim=2, horizon=1, dim=256, depth=4).to(device)
dummy_obs = lambda B: torch.zeros(B, 1, device=device)

def ddpm_loss(model, x0):
    """x0: (B, 2)"""
    B = x0.shape[0]
    # TODO:
    # 1) t ~ 均匀整数 [0, T)；noise ~ N(0, I)
    # 2) xt = q_sample(x0, t, noise)
    # 3) pred = model(xt[:, None, :], t.float() / T, dummy_obs(B))[:, 0]     （网络要 (B, H, A) 形状）
    # 4) return MSE(pred, noise)
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
model = ChunkDenoiser(obs_dim=1, act_dim=2, horizon=1, dim=256, depth=4).to(device)
dummy_obs = lambda B: torch.zeros(B, 1, device=device)

def ddpm_loss(model, x0):
    """x0: (B, 2)"""
    B = x0.shape[0]
    t = torch.randint(0, T, (B,), device=device)
    noise = torch.randn_like(x0)
    xt = q_sample(x0, t, noise)
    pred = model(xt[:, None, :], t.float() / T, dummy_obs(B))[:, 0]
    return ((pred - noise) ** 2).mean()

In [ ]:
# ✅ 训练（GPU 约 20 秒，CPU 约 1~2 分钟）
set_seed(0)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
steps = 4000
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
for step in range(steps):
    loss = ddpm_loss(model, data[torch.randint(0, len(data), (512,), device=device)])
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    if step % 1000 == 0:
        print(f"step {step}  loss {loss.item():.4f}")
assert loss.item() < 0.6
print("通过 ✅")

### 练习 3：反向采样

In [ ]:
@torch.no_grad()
def p_sample_loop(model, n, return_all=False):
    x = torch.randn(n, 2, device=device)
    xs = [x]
    for t in reversed(range(T)):
        eps = model(x[:, None, :], torch.full((n,), t / T, device=device), dummy_obs(n))[:, 0]
        # TODO: 按公式计算均值 mean；t > 0 时加噪声 sqrt(β̃_t) z，t == 0 时 x = mean
        #   β̃_t = β_t (1 - ᾱ_{t-1}) / (1 - ᾱ_t)
        raise NotImplementedError
        xs.append(x)
    return (x, xs) if return_all else x

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
@torch.no_grad()
def p_sample_loop(model, n, return_all=False):
    x = torch.randn(n, 2, device=device)
    xs = [x]
    for t in reversed(range(T)):
        eps = model(x[:, None, :], torch.full((n,), t / T, device=device), dummy_obs(n))[:, 0]
        a, ab, b = alphas[t], alphas_bar[t], betas[t]
        mean = (x - b / (1 - ab).sqrt() * eps) / a.sqrt()
        if t > 0:
            var = b * (1 - alphas_bar[t - 1]) / (1 - ab)
            x = mean + var.sqrt() * torch.randn_like(x)
        else:
            x = mean
        xs.append(x)
    return (x, xs) if return_all else x

In [ ]:
# ✅ 自测：生成的点应该落在 6 个高斯团上
model.eval()
samples, traj = p_sample_loop(model, 3000, return_all=True)
fig, axes = plt.subplots(1, 5, figsize=(14, 2.8))
for ax, k in zip(axes, [0, 50, 80, 95, 100]):
    ax.scatter(*traj[k].cpu().T, s=1, c="C1"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal"); ax.set_title(f"reverse step {k}/{T}")
plt.suptitle("reverse process: noise -> data"); plt.tight_layout(); plt.show()
r = samples.norm(dim=-1)
on_ring = ((r - 1).abs() < 0.25).float().mean().item()
angle = torch.atan2(samples[:, 1], samples[:, 0]) % (2 * math.pi)
cluster = ((angle / (2 * math.pi / 6)).round() % 6).long()
counts = torch.bincount(cluster, minlength=6).float() / len(samples)
print(f"落在圆环附近的比例 {on_ring:.2f}；6 个团的占比 {counts.cpu().numpy().round(2)}")
assert on_ring > 0.85 and counts.min() > 0.08
print("通过 ✅ 所有 6 个模式都被生成出来了（没有模式坍缩）")

---

## 4. 加速采样：DDIM（了解即可）

DDPM 采样要走 $T$ 步（Diffusion Policy 训练用 100 步），对实时控制太慢。**DDIM**（Song et al. 2021）推导出一个**确定性**的采样过程，可以跳步（比如只走 10 步），而且不需要重新训练：

$$
\hat{\mathbf x}_0=\frac{\mathbf x_t-\sqrt{1-\bar\alpha_t}\,\boldsymbol\epsilon_\theta}{\sqrt{\bar\alpha_t}},\qquad
\mathbf x_{t'}=\sqrt{\bar\alpha_{t'}}\,\hat{\mathbf x}_0+\sqrt{1-\bar\alpha_{t'}}\,\boldsymbol\epsilon_\theta\quad(t'<t)
$$

先用预测的噪声估计出"干净的数据" $\hat{\mathbf x}_0$，再按目标时间步重新加上"同样的噪声"。

In [ ]:
@torch.no_grad()
def ddim_sample(model, n, n_steps=10):
    ts = torch.linspace(T - 1, 0, n_steps, device=device).round().long()
    x = torch.randn(n, 2, device=device)
    for i, t in enumerate(ts):
        eps = model(x[:, None, :], torch.full((n,), t.item() / T, device=device), dummy_obs(n))[:, 0]
        x0_hat = (x - (1 - alphas_bar[t]).sqrt() * eps) / alphas_bar[t].sqrt()
        ab_next = alphas_bar[ts[i + 1]] if i + 1 < n_steps else torch.tensor(1.0, device=device)
        x = ab_next.sqrt() * x0_hat + (1 - ab_next).sqrt() * eps
    return x

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, k in zip(axes, [2, 5, 20]):
    s = ddim_sample(model, 3000, k)
    ax.scatter(*s.cpu().T, s=1, c="C2"); ax.set_xlim(-2, 2); ax.set_ylim(-2, 2); ax.set_aspect("equal"); ax.set_title(f"DDIM {k} steps")
plt.tight_layout(); plt.show()

步数太少时（2 步）样本会"糊"在团与团之间。下一课的流匹配从另一个角度出发，路径更"直"，少量步数就能得到好的结果。

---

## 小结

| 公式 | 含义 |
|---|---|
| $\mathbf x_t=\sqrt{\bar\alpha_t}\mathbf x_0+\sqrt{1-\bar\alpha_t}\boldsymbol\epsilon$ | 一步加噪 |
| $\min\|\boldsymbol\epsilon-\boldsymbol\epsilon_\theta(\mathbf x_t,t)\|^2$ | 训练：预测噪声 |
| $\mathbf x_{t-1}=\frac{1}{\sqrt{\alpha_t}}(\mathbf x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\boldsymbol\epsilon_\theta)+\sigma_t\mathbf z$ | 采样：逐步去噪 |
| DDIM | 确定性、可跳步的采样 |

- 扩散模型能表示任意复杂的多峰分布，不会坍缩。
- 代价：采样需要多步迭代。